In [9]:
!pip install pdfplumber
import pdfplumber
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [10]:
def extract_text_from_pdf(filepath):
    with pdfplumber.open(filepath) as pdf:
        text = ""
        for page in pdf.pages:
            t = page.extract_text()
            text += t if t else ""
    return text

In [11]:
job_description = """
We are hiring a Machine Learning Engineer.
The candidate must have strong Python and SQL skills.
Knowledge of TensorFlow, Keras, and deep learning is required.
Experience with data analysis and NLP is a plus.
Strong problem solving and communication skills are essential.
"""

In [12]:
resume_folder = "/content"

resume_files = [
    f for f in os.listdir(resume_folder) if f.endswith(".pdf")
]

print(f"Found {len(resume_files)} resumes")

# Extract text from each resume
resume_texts = {}
for filename in resume_files:
    filepath = os.path.join(resume_folder, filename)
    text = extract_text_from_pdf(filepath)
    resume_texts[filename.replace(".pdf", "")] = text

print("All resumes loaded!")

Found 8 resumes
All resumes loaded!


In [13]:
def semantic_match_score(resume_text, job_text):
    vectorizer = TfidfVectorizer(stop_words='english')
    vectors = vectorizer.fit_transform([job_text, resume_text])
    score = cosine_similarity(vectors[0], vectors[1])[0][0]
    return round(score * 100, 2)

semantic_results = []

for candidate, text in resume_texts.items():
    score = semantic_match_score(text, job_description)
    semantic_results.append({
        "Candidate": candidate,
        "Semantic Score (%)": score
    })

semantic_results.sort(key=lambda x: x["Semantic Score (%)"], reverse=True)

print("=" * 55)
print("     SEMANTIC RANKING — MEANING BASED MATCHING")
print("=" * 55)

for i, r in enumerate(semantic_results, start=1):
    print(f"\nRank #{i}  —  {r['Candidate']}")
    print(f"  Semantic Score : {r['Semantic Score (%)']}%")

     SEMANTIC RANKING — MEANING BASED MATCHING

Rank #1  —  Anurag_gupta
  Semantic Score : 12.02%

Rank #2  —  hamnaAyub
  Semantic Score : 10.59%

Rank #3  —  nambyarRastogi
  Semantic Score : 9.39%

Rank #4  —  AngelaMaria
  Semantic Score : 8.23%

Rank #5  —  akanshaChate
  Semantic Score : 5.27%

Rank #6  —  AdityaSingh
  Semantic Score : 5.13%

Rank #7  —  saurabhgupta
  Semantic Score : 4.15%

Rank #8  —  SiddharthMalhotra
  Semantic Score : 3.49%


In [14]:
skills_database = {
    "python": ["python"],
    "machine learning": ["machine learning", "ml"],
    "deep learning": ["deep learning", "dl"],
    "natural language processing": ["natural language processing", "nlp"],
    "data analysis": ["data analysis", "data analytics"],
    "artificial intelligence": ["artificial intelligence", "ai"],
    "sql": ["sql", "mysql", "postgresql"],
    "java": ["java"],
    "javascript": ["javascript", "js"],
    "html": ["html"],
    "css": ["css"],
    "excel": ["excel", "ms excel"],
    "communication": ["communication"],
    "teamwork": ["teamwork", "team player"],
    "problem solving": ["problem solving"],
    "data science": ["data science"],
    "tensorflow": ["tensorflow"],
    "keras": ["keras"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "power bi": ["power bi"],
    "tableau": ["tableau"],
    "c++": ["c++"],
    "r": [" r ", "r programming"],
}

def extract_skills(text, skills_db):
    text_lower = text.lower()
    found = []
    for skill, synonyms in skills_db.items():
        for synonym in synonyms:
            if synonym in text_lower:
                found.append(skill)
                break
    return found

def keyword_score(resume_text, job_text):
    job_skills = extract_skills(job_text, skills_database)
    resume_skills = extract_skills(resume_text, skills_database)
    matched = [s for s in resume_skills if s in job_skills]
    return round((len(matched) / len(job_skills)) * 100, 2) if job_skills else 0

# Calculate hybrid score for all candidates
hybrid_results = []

for candidate, text in resume_texts.items():
    k_score = keyword_score(text, job_description)
    s_score = semantic_match_score(text, job_description)

    # 50% keyword + 50% semantic
    hybrid = round((k_score * 0.5) + (s_score * 0.5), 2)

    hybrid_results.append({
        "Candidate": candidate,
        "Keyword Score (%)": k_score,
        "Semantic Score (%)": s_score,
        "Hybrid Score (%)": hybrid
    })

hybrid_results.sort(key=lambda x: x["Hybrid Score (%)"], reverse=True)

print("=" * 65)
print("        FINAL HYBRID RANKING — KEYWORD + SEMANTIC")
print("=" * 65)

for i, r in enumerate(hybrid_results, start=1):
    print(f"\nRank #{i}  —  {r['Candidate']}")
    print(f"  Keyword Score  : {r['Keyword Score (%)']}%")
    print(f"  Semantic Score : {r['Semantic Score (%)']}%")
    print(f"  Hybrid Score   : {r['Hybrid Score (%)']}%")

# Clean table view
df = pd.DataFrame(hybrid_results).reset_index(drop=True)
df.index += 1
df

        FINAL HYBRID RANKING — KEYWORD + SEMANTIC

Rank #1  —  hamnaAyub
  Keyword Score  : 60.0%
  Semantic Score : 10.59%
  Hybrid Score   : 35.3%

Rank #2  —  AngelaMaria
  Keyword Score  : 60.0%
  Semantic Score : 8.23%
  Hybrid Score   : 34.12%

Rank #3  —  akanshaChate
  Keyword Score  : 60.0%
  Semantic Score : 5.27%
  Hybrid Score   : 32.64%

Rank #4  —  saurabhgupta
  Keyword Score  : 50.0%
  Semantic Score : 4.15%
  Hybrid Score   : 27.08%

Rank #5  —  nambyarRastogi
  Keyword Score  : 40.0%
  Semantic Score : 9.39%
  Hybrid Score   : 24.7%

Rank #6  —  SiddharthMalhotra
  Keyword Score  : 30.0%
  Semantic Score : 3.49%
  Hybrid Score   : 16.74%

Rank #7  —  AdityaSingh
  Keyword Score  : 20.0%
  Semantic Score : 5.13%
  Hybrid Score   : 12.56%

Rank #8  —  Anurag_gupta
  Keyword Score  : 10.0%
  Semantic Score : 12.02%
  Hybrid Score   : 11.01%


,Candidate,Keyword Score (%),Semantic Score (%),Hybrid Score (%)
1,hamnaAyub,60.0,10.59,35.30
2,AngelaMaria,60.0,8.23,34.12
3,akanshaChate,60.0,5.27,32.64
4,saurabhgupta,50.0,4.15,27.08
5,nambyarRastogi,40.0,9.39,24.70
6,SiddharthMalhotra,30.0,3.49,16.74
7,AdityaSingh,20.0,5.13,12.56
8,Anurag_gupta,10.0,12.02,11.01
